In [ ]:

# ======================
# Parameters 
# ========================
INPUT_CSV = r'C:\Users\Vivian\Documents\CLAM\CLAM\dataset_csv\fa_vs_pt.csv'
NAME      = 'cross-val/FA_PT'   # Name prefix for split folders
OUTDIR    = r'C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits'               # Root output directory
K_FOLDS   = 5                      # Number of folds
VAL_PROP  = 0.2                    # Proportion of val from train+val groups (e.g., 0.2)
SEED      = 42                     # Random seed
LABEL_MAP = None                   # Optional dict, e.g., {'0': 'FA', '1': 'PT'} or {0:'FA',1:'PT'}
CASE_COL  = 'case_id'
SLIDE_COL = 'slide_id'
LABEL_COL = 'label'

# If don't want a validation split, set VAL_PROP = 0.0 and ignore the generated val.csv.


In [2]:

import os
from pathlib import Path
import pandas as pd
import numpy as np
from collections import Counter, defaultdict

def _get_stratified_group_kfold():
    """Try to import StratifiedGroupKFold (sklearn >= 1.1)."""
    try:
        from sklearn.model_selection import StratifiedGroupKFold
        return StratifiedGroupKFold
    except Exception:
        return None


In [7]:

def derive_group_labels(df: pd.DataFrame, case_col: str, label_col: str):
    """
    Derive a single label per patient (group) for stratification.
    Uses majority label across that patient's slides (ties resolved deterministically).
    Returns: dict case_id -> group_label (as string).
    """
    group_labels = {}
    for cid, sub in df.groupby(case_col):
        counts = Counter(sub[label_col].astype(str).tolist())
        # Majority; tie-break by lexicographic to be deterministic
        majority_label = sorted(counts.items(), key=lambda x: (-x[1], x[0]))[0][0]
        group_labels[cid] = majority_label
    return group_labels


def stratified_group_kfold_split(cases: np.ndarray, y_groups: np.ndarray, k: int, seed: int):
    """
    Returns list of (trainval_idx, test_idx) over cases (group-level).
    If StratifiedGroupKFold exists, use it. Otherwise emulate with a greedy bin-packing.
    """
    SGKF = _get_stratified_group_kfold()
    if SGKF is not None:
        sgkf = SGKF(n_splits=k, shuffle=True, random_state=seed)
        splits = []
        X_dummy = np.zeros((len(cases), 1))
        for tr_idx, te_idx in sgkf.split(X_dummy, y_groups, groups=cases):
            splits.append((tr_idx, te_idx))
        return splits

    # Fallback: balanced greedy assignment of cases into k buckets maintaining label proportions
    rng = np.random.RandomState(seed)
    unique_labels, counts = np.unique(y_groups, return_counts=True)

    # Shuffle cases inside each label
    label_to_cases = defaultdict(list)
    for c, y in zip(cases, y_groups):
        label_to_cases[y].append(c)
    for y in label_to_cases:
        rng.shuffle(label_to_cases[y])

    # Initialize k folds with label counters
    folds = [defaultdict(int) for _ in range(k)]
    fold_cases = [set() for _ in range(k)]

    # Assign cases label by label to the fold with the lowest count for that label
    for y in unique_labels:
        for c in label_to_cases[y]:
            # pick fold with smallest count for label y, tie-break by smallest total size
            best_fold = None
            best_tuple = None
            for i in range(k):
                label_count = folds[i][y]
                total = sum(folds[i].values())
                t = (label_count, total, i)
                if (best_tuple is None) or (t < best_tuple):
                    best_tuple = t
                    best_fold = i
            folds[best_fold][y] += 1
            fold_cases[best_fold].add(c)

    splits = []
    all_cases = set(cases.tolist())
    for i in range(k):
        test_cases = np.array(sorted(list(fold_cases[i])))
        trainval_cases = np.array(sorted(list(all_cases - fold_cases[i])))
        tr_idx = np.where(np.isin(cases, trainval_cases))[0]
        te_idx = np.where(np.isin(cases, test_cases))[0]
        splits.append((tr_idx, te_idx))
    return splits


def stratified_group_val_split(trainval_cases: np.ndarray, y_groups: np.ndarray, val_prop: float, seed: int):
    """
    Split trainval_cases into train and val (stratified by group label).
    Uses sklearn StratifiedShuffleSplit over groups if available; else manual.
    """
    try:
        from sklearn.model_selection import StratifiedShuffleSplit
        sss = StratifiedShuffleSplit(n_splits=1, test_size=val_prop, random_state=seed)
        X_dummy = np.zeros((len(trainval_cases), 1))
        (train_idx,), (val_idx,) = next(sss.split(X_dummy, y_groups))
        return trainval_cases[train_idx], trainval_cases[val_idx]
    except Exception:
        # Manual deterministic split per label
        rng = np.random.RandomState(seed)
        train_cases = []
        val_cases = []
        for y in np.unique(y_groups):
            mask = (y_groups == y)
            cases_y = trainval_cases[mask]
            rng.shuffle(cases_y)
            n_val = max(1, int(round(len(cases_y) * val_prop))) if len(cases_y) > 1 else 0
            val_cases.extend(cases_y[:n_val])
            train_cases.extend(cases_y[n_val:])
        return np.array(sorted(train_cases)), np.array(sorted(val_cases))

# Fix for Python < 3.10 (no PEP 604 union types)
from typing import Optional

def write_split_csv(df: pd.DataFrame, out_path: Path, slide_col: str, label_col: str, label_map: Optional[dict] = None):
    sub = df[[slide_col, label_col]].copy()
    if label_map is not None:
        # normalize keys to strings so both {0:'FA'} and {'0':'FA'} work
        norm_map = {str(k): v for k, v in label_map.items()}
        sub[label_col] = sub[label_col].astype(str).map(lambda x: norm_map.get(x, x))
    # Ensure exact column names and order
    sub.columns = ["slide_id", "label"]
    sub = sub.sort_values(by=["slide_id"]).reset_index(drop=True)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sub.to_csv(out_path, index=False)



In [8]:

# ========================
# Build splits
# ========================

df = pd.read_csv(INPUT_CSV)
required_cols = {CASE_COL, SLIDE_COL, LABEL_COL}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in {INPUT_CSV}: {missing}")

# Derive patient-level labels for stratification
group_label_map = derive_group_labels(df, case_col=CASE_COL, label_col=LABEL_COL)
cases = np.array(sorted(group_label_map.keys()))
y_groups = np.array([group_label_map[c] for c in cases])

# Build outer k-fold splits (by patient)
splits = stratified_group_kfold_split(cases, y_groups, k=K_FOLDS, seed=SEED)

out_root = Path(OUTDIR)

reports = []
for k_idx, (trval_idx, te_idx) in enumerate(splits):
    test_cases = cases[te_idx]
    trainval_cases = cases[trval_idx]

    # Build y for trainval cases for val split
    y_trainval = np.array([group_label_map[c] for c in trainval_cases])

    # Inner split: train vs val (by patient)
    if VAL_PROP and VAL_PROP > 0.0:
        train_cases, val_cases = stratified_group_val_split(trainval_cases, y_trainval, val_prop=VAL_PROP, seed=SEED + k_idx)
    else:
        train_cases, val_cases = trainval_cases, np.array([], dtype=trainval_cases.dtype)

    # Materialize slide-level rows
    is_train = df[CASE_COL].isin(train_cases)
    is_val   = df[CASE_COL].isin(val_cases)
    is_test  = df[CASE_COL].isin(test_cases)

    df_train = df[is_train].copy()
    df_val   = df[is_val].copy()
    df_test  = df[is_test].copy()

    fold_dir = out_root / f"{NAME}_k={k_idx}"
    write_split_csv(df_train, fold_dir / "train.csv", SLIDE_COL, LABEL_COL, LABEL_MAP)
    if len(df_val) > 0:
        write_split_csv(df_val,   fold_dir / "val.csv",   SLIDE_COL, LABEL_COL, LABEL_MAP)
    else:
        # Still create an empty val.csv for consistency
        (fold_dir / "val.csv").parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(columns=["slide_id", "label"]).to_csv(fold_dir / "val.csv", index=False)
    write_split_csv(df_test,  fold_dir / "test.csv",  SLIDE_COL, LABEL_COL, LABEL_MAP)

    reports.append({
        'fold': k_idx,
        'train_slides': len(df_train),
        'val_slides': len(df_val),
        'test_slides': len(df_test),
        'train_patients': df_train[CASE_COL].nunique(),
        'val_patients': df_val[CASE_COL].nunique(),
        'test_patients': df_test[CASE_COL].nunique(),
    })

pd.DataFrame(reports)


,fold,train_slides,val_slides,test_slides,train_patients,val_patients,test_patients
0,0,155,38,47,93,23,29
1,1,157,35,48,93,23,29
2,2,157,38,45,93,23,29
3,3,156,35,49,93,23,29
4,4,149,40,51,93,23,29


In [10]:
from pathlib import Path
import pandas as pd

SPLITS_ROOT = Path(r'C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\cross-val')  # change if needed
MAP = {'0': 'FA', '1': 'PT', 0: 'FA', 1: 'PT'}

def relabel_csv(p: Path):
    df = pd.read_csv(p)
    if 'label' in df.columns:
        df['label'] = df['label'].map(lambda x: MAP.get(x, MAP.get(str(x), x)))
        df.to_csv(p, index=False)

for csv in SPLITS_ROOT.rglob("*.csv"):
    relabel_csv(csv)

print("Done relabeling to FA/PT.")


Done relabeling to FA/PT.



### Notes
- For downstream code expecting feature directories, ensure your paths end with `feats_h5/` (for HDF5 features) or `feats_pt/` (for PyTorch features).
- To map numeric labels to strings (e.g., `0->FA`, `1->PT`), set:
  ```python
  LABEL_MAP = {'0': 'FA', '1': 'PT'}  # or {0: 'FA', 1: 'PT'}
  ```
- To run **pure 5-fold CV** with no validation split, set `VAL_PROP = 0.0`.
